# ResNet Feature Extraction cho Video Search

## Mục đích
Dùng **ResNet50** (pretrained trên ImageNet) để trích xuất **semantic feature vector** từ ảnh/keyframe.

### So sánh với HSV + HOG hiện tại:
| | HSV + HOG | ResNet50 |
|---|---|---|
| Loại feature | Low-level (màu, cạnh) | High-level (ngữ nghĩa) |
| Vector size | ~128 + hàng ngàn chiều | 2048 chiều |
| Hiểu nội dung | ❌ Không | ✅ Có |
| Tốc độ | Nhanh (CPU) | Chậm hơn (GPU tốt hơn) |

## 1. Import thư viện

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import cv2
import os
from pathlib import Path

# Kiểm tra GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2. Tạo ResNet Feature Extractor

### Giải thích:
- **ResNet50** gồm nhiều layer: conv1 → bn1 → relu → maxpool → layer1~4 → avgpool → fc
- Layer cuối `fc` là fully-connected phân loại 1000 class ImageNet → **BỎ layer này**
- Lấy output từ `avgpool` → vector **2048 chiều** chứa semantic features
- `weights=IMAGENET1K_V2`: dùng trọng số pretrained trên ImageNet (tốt nhất)
- `model.eval()`: chuyển sang chế độ inference (tắt dropout, batch norm dùng running stats)

In [ ]:
def create_resnet_extractor():
    # Load ResNet50 pretrained trên ImageNet
    resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    
    # Bỏ layer fc cuối (layer phân loại 1000 class)
    # nn.Identity() = layer "không làm gì" → output giữ nguyên từ avgpool
    resnet.fc = nn.Identity()
    
    # Chuyển sang chế độ evaluation (tắt dropout, batch norm ổn định)
    resnet.eval()
    
    # Chuyển model lên GPU nếu có
    resnet = resnet.to(device)
    
    return resnet

model = create_resnet_extractor()
print(f'Model loaded. Output: 2048D feature vector')

## 3. Preprocessing ảnh

### Giải thích từng bước transform:
1. **Resize(256)**: Thu nhỏ ảnh sao cho cạnh ngắn = 256px (giữ tỷ lệ)
2. **CenterCrop(224)**: Cắt vùng trung tâm 224×224 (kích thước chuẩn ImageNet)
3. **ToTensor()**: Chuyển ảnh PIL (0-255, H×W×C) → tensor (0.0-1.0, C×H×W)
4. **Normalize(mean, std)**: Chuẩn hóa theo thống kê ImageNet
   - mean=[0.485, 0.456, 0.406] → trung bình RGB của ImageNet
   - std=[0.229, 0.224, 0.225] → độ lệch chuẩn RGB của ImageNet
   - Công thức: `pixel = (pixel - mean) / std`
   - Mục đích: Đưa input về cùng phân phối với dữ liệu training

In [ ]:
# Transform pipeline: chuẩn bị ảnh đầu vào cho ResNet
preprocess = transforms.Compose([
    transforms.Resize(256),           # Resize cạnh ngắn → 256px
    transforms.CenterCrop(224),       # Cắt trung tâm 224×224
    transforms.ToTensor(),            # PIL → Tensor (0-1, C×H×W)
    transforms.Normalize(             # Chuẩn hóa theo ImageNet
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

print('Preprocessing pipeline ready')

## 4. Hàm trích xuất feature từ 1 ảnh

### Giải thích:
- `torch.no_grad()`: Tắt tính gradient (tiết kiệm RAM, chỉ cần inference)
- `.unsqueeze(0)`: Thêm batch dimension [C,H,W] → [1,C,H,W] (model cần batch)
- `.squeeze().cpu().numpy()`: Bỏ batch dim, chuyển về CPU, thành numpy array
- **L2 normalize**: Chia vector cho norm để ||v|| = 1 → cosine similarity = dot product

In [ ]:
def extract_feature_from_pil(pil_image, model):
    
    # Áp dụng preprocessing
    input_tensor = preprocess(pil_image)  # [3, 224, 224]
    
    # Thêm batch dimension: [3,224,224] → [1,3,224,224]
    input_batch = input_tensor.unsqueeze(0).to(device)
    
    # Forward pass (không cần gradient vì chỉ inference)
    with torch.no_grad():
        features = model(input_batch)  # [1, 2048]
    
    # Chuyển về numpy: bỏ batch dim, về CPU
    vec = features.squeeze().cpu().numpy()  # (2048,)
    
    # L2 normalize: ||vec|| = 1 → cosine similarity = dot product
    norm = np.linalg.norm(vec)
    if norm > 0:
        vec = vec / norm
    
    return vec


def extract_feature_from_cv2(bgr_frame, model):
    # Chuyển BGR (OpenCV) → RGB (PIL)
    rgb = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(rgb)
    return extract_feature_from_pil(pil_img, model)


print('Feature extraction functions ready')

## 5. Test: Trích xuất feature từ ảnh query

Dùng ảnh test trong `public/test_queries/` để kiểm tra.

In [ ]:
# Đường dẫn project
PROJECT_ROOT = Path().resolve().parent.parent
TEST_DIR = PROJECT_ROOT / 'public' / 'test_queries'

# Load và trích xuất feature cho tất cả ảnh test
test_features = {}
for img_path in sorted(TEST_DIR.glob('*.jpg')):
    pil_img = Image.open(img_path).convert('RGB')
    vec = extract_feature_from_pil(pil_img, model)
    test_features[img_path.stem] = vec
    print(f'{img_path.name}: vector shape = {vec.shape}, norm = {np.linalg.norm(vec):.4f}')

print(f'\nExtracted features for {len(test_features)} images')

## 6. So sánh similarity giữa các ảnh test

Vì vector đã L2 normalize → **cosine similarity = dot product**

In [ ]:
def cosine_similarity(vec_a, vec_b):
    dot = np.dot(vec_a, vec_b)
    norm = np.linalg.norm(vec_a) * np.linalg.norm(vec_b)
    if norm < 1e-9:
        return 0.0
    return float(dot / norm)


# So sánh tất cả cặp ảnh
names = list(test_features.keys())
print('Cosine Similarity Matrix:')
print(f'{"":>20s}', end='')
for n in names:
    print(f'{n:>18s}', end='')
print()

for i, n1 in enumerate(names):
    print(f'{n1:>20s}', end='')
    for j, n2 in enumerate(names):
        sim = cosine_similarity(test_features[n1], test_features[n2])
        print(f'{sim:>18.4f}', end='')
    print()

## 7. Trích xuất features cho keyframes của video

Hàm này dùng để thay thế `MFrame.compute_his()` + `compute_hog()` + `compute_vec()`
bằng ResNet feature extraction cho mỗi keyframe.

In [ ]:
def extract_keyframe_features(video_path, model, max_keyframes=15):
    """
    Tách keyframes từ video và trích xuất ResNet features.
    
    Dùng scene detection đơn giản (histogram diff) để tách keyframe,
    sau đó chạy ResNet cho từng keyframe.
    
    Args:
        video_path: Đường dẫn video
        model: ResNet50 extractor
        max_keyframes: Số keyframe tối đa
    
    Returns:
        list[dict]: Mỗi item gồm {frame_idx, timestamp, vector}
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f'Cannot open: {video_path}')
    
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    results = []
    prev_hist = None
    frame_idx = 0
    THRESHOLD = 0.4
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Tính histogram HSV đơn giản để phát hiện scene change
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        hist = cv2.calcHist([hsv], [0,1,2], None, [8,4,4], [0,180,0,256,0,256])
        hist = cv2.normalize(hist, hist).flatten()
        
        is_keyframe = False
        if prev_hist is None:
            is_keyframe = True
        else:
            diff = 1.0 - float(np.dot(hist, prev_hist) / 
                              (np.linalg.norm(hist) * np.linalg.norm(prev_hist) + 1e-9))
            is_keyframe = diff > THRESHOLD
        
        if is_keyframe:
            # Dùng ResNet trích xuất feature (thay cho HSV+HOG)
            vec = extract_feature_from_cv2(frame, model)
            results.append({
                'frame_idx': frame_idx,
                'timestamp': round(frame_idx / fps, 3),
                'vector': vec,
            })
            prev_hist = hist
            
            if len(results) >= max_keyframes:
                break
        
        frame_idx += 1
    
    cap.release()
    return results

print('Keyframe extraction function ready')

## Tóm tắt Pipeline

```
                    UPLOAD VIDEO
                        |
            [Scene Detection - Histogram]
                        |
                   Keyframes (ảnh)
                        |
              [ResNet50 - Feature Extraction]
                        |
               Vector 2048D (L2 normalized)
                        |
                  Lưu vào DB (pgvector)


                   SEARCH (ảnh query)
                        |
              [ResNet50 - Feature Extraction]
                        |
               Vector 2048D query
                        |
          [Cosine Similarity với tất cả keyframes]
                        |
              Top-5 video tương đồng nhất
```